In [1]:
import geopandas as gpd
from pathlib import Path

In [2]:
property_gdf = gpd.read_parquet("../../data/processed/nsw_property/property.parquet")
zoning_gdf = gpd.read_parquet("../../data/processed/nsw_zoning/land_zoning.parquet")

In [3]:
# Basic information
print("Property rows:", len(property_gdf))
print("Zoning rows:", len(zoning_gdf))

print("Property CRS:", property_gdf.crs)
print("Zoning CRS:", zoning_gdf.crs)

print(property_gdf.columns)
print(zoning_gdf.columns)

Property rows: 4198396
Zoning rows: 112245
Property CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "dire

In [4]:
# Filter property main table
property_main = property_gdf.copy()
print(len(property_main))
property_main["propertytype"].value_counts(dropna=False)

4198396


propertytype
1    4193334
6       5062
Name: count, dtype: int64

In [5]:
# only keep zoning core fields
zoning_keep = zoning_gdf[
    ["OBJECTID", "EPI_NAME", "LGA_NAME", "LAY_CLASS", "SYM_CODE", "PURPOSE", "EPI_TYPE", "geometry"]
].copy()

In [6]:
# CRS align
if property_main.crs != zoning_keep.crs:
    property_main = property_main.to_crs(zoning_keep.crs)

In [7]:
property_sample = property_main.sample(50000, random_state=42).copy()
print(len(property_sample))

50000


In [8]:
property_sample["propertytype"].value_counts(dropna=False)

propertytype
1    49950
6       50
Name: count, dtype: int64

In [9]:
joined = gpd.sjoin(
    property_sample,
    zoning_keep,
    how="left",
    predicate="intersects"
)

In [10]:
print("Joined rows:", len(joined))
joined[
    ["RID", "propertytype", "address", "Shape__Area", "SYM_CODE", "LAY_CLASS", "EPI_NAME", "LGA_NAME"]
].head(10)

Joined rows: 109666


,RID,propertytype,address,Shape__Area,SYM_CODE,LAY_CLASS,EPI_NAME,LGA_NAME
56554,70231,1,49 AEOLUS AVENUE RYDE,872.277868,R2,Low Density Residential,Ryde Local Environmental Plan 2014,RYDE
56554,70231,1,49 AEOLUS AVENUE RYDE,872.277868,R2,Low Density Residential,Ryde Local Environmental Plan 2014,RYDE
2671533,3755139,1,3 CRAIGIE AVENUE PADSTOW,732.668163,R2,Low Density Residential,Canterbury-Bankstown Local Environmental Plan ...,CANTERBURY-BANKSTOWN
4182802,7500698,1,108/3 EARL PLACE POTTS POINT,839.802288,MU1,Mixed Use,Sydney Local Environmental Plan 2012,SYDNEY
4182802,7500698,1,108/3 EARL PLACE POTTS POINT,839.802288,MU1,Mixed Use,Sydney Local Environmental Plan 2012,SYDNEY
4182802,7500698,1,108/3 EARL PLACE POTTS POINT,839.802288,E1,Local Centre,Sydney Local Environmental Plan 2012,SYDNEY
4182802,7500698,1,108/3 EARL PLACE POTTS POINT,839.802288,E1,Local Centre,Sydney Local Environmental Plan 2012,SYDNEY
2963947,4681038,1,43/36-44 FONTENOY ROAD MACQUARIE PARK,15276.563641,C2,Environmental Conservation,Ryde Local Environmental Plan 2014,RYDE
2963947,4681038,1,43/36-44 FONTENOY ROAD MACQUARIE PARK,15276.563641,C2,Environmental Conservation,Ryde Local Environmental Plan 2014,RYDE
2963947,4681038,1,43/36-44 FONTENOY ROAD MACQUARIE PARK,15276.563641,R4,High Density Residential,Ryde Local Environmental Plan 2014,RYDE


In [11]:
joined["SYM_CODE"].isna().mean()

np.float64(0.0012218919263946894)

In [12]:
joined["SYM_CODE"].value_counts(dropna=False).head(20)

SYM_CODE
R2     32581
R1     11869
R3      9941
SP2     7290
R4      6422
RE1     5977
MU1     5194
RU1     4765
E1      3298
C4      2492
C2      2354
RU5     2015
E4      1955
R5      1608
RU2     1564
C3      1449
E2      1296
E3       982
C1       854
RE2      798
Name: count, dtype: int64

In [13]:
joined["RID"].duplicated().mean()

np.float64(0.5440701767183995)

In [14]:
joined.groupby("propertytype")["SYM_CODE"].apply(lambda s: s.isna().mean()).sort_index()

propertytype
1    0.001214
6    0.006623
Name: SYM_CODE, dtype: float64

In [19]:
print(joined.groupby("propertytype")["Shape__Area"].describe())

                 count          mean           std        min          25%  \
propertytype                                                                 
1             109515.0  5.559400e+05  1.400967e+07   0.484096   923.512813   
6                151.0  5.550234e+06  1.413102e+07  50.238020  4912.373099   

                       50%           75%           max  
propertytype                                            
1              1675.641095  8.499395e+03  9.411130e+08  
6             36067.894547  3.604425e+06  6.799978e+07  


In [16]:
dup_joined = joined[joined["RID"].duplicated(keep=False)].copy()
dup_joined["propertytype"].value_counts(dropna=False)

propertytype
1    93336
6      138
Name: count, dtype: int64

In [17]:
joined["lot_size_proxy_sqm"] = joined["Shape__Area"]

site_sample = joined[
    [
        "RID",
        "gurasid",
        "propertytype",
        "valnetpropertystatus",
        "valnetpropertytype",
        "dissolveparcelcount",
        "valnetlotcount",
        "propid",
        "superlot",
        "address",
        "housenumber",
        "urbanity",
        "Shape__Area",
        "Shape__Length",
        "lot_size_proxy_sqm",
        "SYM_CODE",
        "LAY_CLASS",
        "EPI_NAME",
        "LGA_NAME",
        "EPI_TYPE",
        "geometry",
    ]
].copy()

In [20]:
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

site_sample.to_parquet(
    "../../data/processed/site_features/property_with_zoning_sample_all_types.parquet",
    index=False
)